In [1]:
from astropy.table import Table, join
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from matplotlib import cm
from matplotlib.lines import Line2D
import os

import time
from numpy.random import default_rng

import healpy as hp
from healpy.newvisufunc import projview, newprojplot
import pandas as pd

import astropy.cosmology
from astropy.coordinates import SkyCoord
from astropy import units as u
from astropy.table import Table, join

import sys
sys.path.insert(0,'../code')
%load_ext autoreload
%autoreload 2
import utils
import generate_random
import correlations
import masks
import maps

%matplotlib

Using matplotlib backend: <object object at 0x111b5c790>


In [2]:
NSIDE = 64
area_per_pixel = hp.nside2pixarea(NSIDE, degrees=True)
print(f"Area per pixel: {area_per_pixel:.3f} deg")

Area per pixel: 0.839 deg


In [3]:
G_hi = 20.5
G_lo = 20.0

In [4]:
quants = {}

### Volume calculations

In [5]:
def v_shells_of_z(z_bins, fsky, cosmo):
    
    v_at_z_bins = np.empty(len(z_bins), dtype=u.Quantity)
    for i in range(len(z_bins)):
        v = cosmo.comoving_volume(z_bins[i])
        v_at_z_bins[i] = v.to(u.Gpc**3)
    v_at_z_bins *= fsky
    v_shells = v_at_z_bins[1:] - v_at_z_bins[:-1] 
    
    # ALT METHOD, same answer
#     z_bins_avg = 0.5*(z_bins[:-1] + z_bins[1:])
#     dvol = cosmo.differential_comoving_volume(z_bins_avg)  # THIS GETS  VOL PER UNIT REDSHIFT PER UNIT SOLID ANGLE
#     dz = z_bins[1:] - z_bins[:-1]
#     dvol = (4.*np.pi)*u.sr * dz * dvol # THIS GETS THE TOTAL VOLUME WITHIN EACH BIN OF REDSHIFT
#     v_shells = dvol.to(u.Gpc**3) * fsky
    
    return np.array(v_shells)

In [6]:
def ndens_of_z(z_arr, z_bins, fsky, cosmo):
    
    v_shells = v_shells_of_z(z_bins, fsky, cosmo)
    ndens = []
    for i in range(len(z_bins)-1):
        N_inbin = np.sum((z_arr >= z_bins[i]) & (z_arr < z_bins[i+1]))
        ndens_inbin = N_inbin/v_shells[i]
        ndens.append(ndens_inbin)
        
    # ALT METHOD, same answer
#     N_zhist = np.histogram(z_arr,bins=z_bins) 
#     ndens = N_zhist[0]/v_shells
    return ndens

In [7]:
# eqn 1.7.32 of https://arxiv.org/pdf/1606.00180.pdf
def volume_effective_Gpcperh(z_arr, z_bins, cosmo, fsky, P0):
    
    ndens_at_z_bins = ndens_of_z(z_arr, z_bins, fsky, cosmo)
    ndens_at_z_bins = np.array(ndens_at_z_bins, dtype=u.Quantity)

    nPs = np.array([n*P0 for n in ndens_at_z_bins])
    prefacs = (nPs /(1 + nPs))**2
    v_shells = v_shells_of_z(z_bins, fsky, cosmo)    
    v_Gpc = np.sum(prefacs*v_shells)

    v_Gpc = v_Gpc.to(u.Gpc**3) # this should be just in Gpc
    v_Gpcperh = v_Gpc * cosmo.h**3 # (Gpc) -> (Gpc/h)^3, mult by h^3 
    print(f"Sky fraction is {fsky:.3f}")
    print(f"Effective volume is {v_Gpc:.3f} = {v_Gpcperh.value:.3f} (Gpc/h)^3")
    return v_Gpcperh.value

In [8]:
from astropy.cosmology import Planck18
cosmo = Planck18
area_allsky = 41252.96125#*(u.deg**2)
P0 = 4e4 * u.Mpc**3 / cosmo.h**3 

In [9]:
fn_dustmap = f'../data/maps/selection_function_template_maps/map_dust_NSIDE{NSIDE}.npy'
map_dust = maps.get_dust_map(NSIDE=NSIDE, R=3.1, fn_map=fn_dustmap)

Av_hi = 0.5
quants['Avhi'] = Av_hi

Dustmap already exists, loading from ../data/maps/selection_function_template_maps/map_dust_NSIDE64.npy


## Compute for DESI

In [10]:
def get_agn_maskbits(file):
    import yaml
    from desiutil_bitmask import BitMask
    file_yaml = open(file, 'r')
    yaml_defs = yaml.safe_load(file_yaml)
    
    AGN_MASKBITS = BitMask('AGN_MASKBITS', yaml_defs)
    OPT_UV_TYPE = BitMask('OPT_UV_TYPE', yaml_defs)
    IR_TYPE = BitMask('IR_TYPE', yaml_defs)
    
    return AGN_MASKBITS, OPT_UV_TYPE, IR_TYPE

In [11]:
fn_desi_lite = "../data/agnqso_desi_QN_lite.fits"

# so we don't have to keep reading in big file
if os.path.exists(fn_desi_lite):
    print(f"Reading {fn_desi_lite}")
    tab_desi_qn_lite = Table.read(fn_desi_lite) 
    print(len(tab_desi_qn_lite))
else:
    overwrite = True
    fn_desi = "../data/agnqso_desi.fits"
    tab_desi = Table.read(fn_desi) 
    print(f"Computing {fn_desi_lite}")

    AGN_MASKBITS, OPT_UV_TYPE, IR_TYPE = get_agn_maskbits('../data/agnmask.yaml')
    sel = (tab_desi['AGN_MASKBITS'] & AGN_MASKBITS['QN'] != 0)
    tab_desi_qn = tab_desi[sel]
    print(len(tab_desi), len(tab_desi_qn))

    tab_desi_qn_lite = tab_desi_qn[['TARGETID', 'Z', 'ZWARN', 'SPECTYPE', 'TARGET_RA', 'TARGET_DEC']]

    tab_desi_qn_lite.write(fn_desi_lite, overwrite=overwrite)

Computing ../data/agnqso_desi_QN_lite.fits
17995599 1418370


In [12]:
i_zgood = tab_desi_qn_lite['ZWARN']==0
tab_desi_good = tab_desi_qn_lite[i_zgood]

#i_Glim_desi = r_sdss_to_G_gaia(r_mag_eboss) < G_hi

In [13]:
len(tab_desi_good)

1400811

In [14]:
# don't have magnitude data for desi in this table, so ignore for now
# tables_Glim = {'desi_Glim': tab_desi_Glim,
#                 }

tables_nolim = {'desi': tab_desi_good,
                }

#tables_all = tables_Glim
#tables_all.update(tables_nolim)
tables_all = tables_nolim

zlabels = {'desi': 'Z',
            }

survey_names = {'desi': 'DESI DR1',
                }

radec_names = {'desi': ['TARGET_RA', 'TARGET_DEC'],
                }

In [15]:
tnames = ['desi']
tables_nodust = {}
for name_full in tnames:
    table = tables_all[name_full]
    name = name_full if 'Glim' not in name_full else name_full.split('_Glim')[0]
    
    ra_name, dec_name = radec_names[name]
    pixel_indices = hp.ang2pix(NSIDE, table[ra_name], table[dec_name], lonlat=True)
    NPIX = hp.nside2npix(NSIDE)
    counts_by_pixel = np.bincount(pixel_indices, minlength=NPIX)
    i_keep = (map_dust[pixel_indices] < Av_hi) & (counts_by_pixel[pixel_indices]>0)
    i_pix_keep = (map_dust < Av_hi) & (counts_by_pixel>0)
    print(name_full, np.sum(i_keep), len(i_keep), f'{np.sum(i_keep)/len(i_keep):.3}')
    tables_nodust[name_full] = table[i_keep]

desi 1395876 1400811 0.996


In [16]:
N_arr, area_arr, nbar_arr, fsky_arr, veff_arr, vspan_arr, zmed_arr = [], [], [], [], [], [], []
fskys = {}

set_dust_thresh = False

tnames = ['desi']
for name_full in tnames:
    
    if set_dust_thresh:
        table = tables_nodust[name_full]
    else:
        table = tables_all[name_full]
    
    name = name_full if 'Glim' not in name_full else name_full.split('_Glim')[0]
    
    ra_name, dec_name = radec_names[name]
    pixel_indices = hp.ang2pix(NSIDE, table[ra_name], table[dec_name], lonlat=True)
    NPIX = hp.nside2npix(NSIDE)
    counts_by_pixel = np.bincount(pixel_indices, minlength=NPIX)
    
    # keep only pixels w data. for dust case, already done above, but redoing doesn't hurt
    i_pix_keep = (counts_by_pixel>0)
        
    print(f'Area from {np.sum(i_pix_keep)}/{len(i_pix_keep)} pixels')
    
    N = len(table)
    quants[f'N_{name}'] = rf'{N:,}'
    
    #area = areas[name]
    area = np.sum(i_pix_keep) * area_per_pixel 
    quants[f'area_{name}'] = rf'{area:.2f} deg$^2$'
    fsky = area/area_allsky
    fskys[name] = fsky
    quants[f'fsky_{name}'] = f'{fsky:.2f}'
    
    nbar = len(table)/area
    print(f'nbar = {nbar:.2f}')
    quants[f'nbar_{name}'] = rf'{nbar:.2f} deg$^{-2}$'

    z_arr = table[zlabels[name]]
    z_min, z_max = np.min(z_arr), np.max(z_arr)
    z_bins = np.arange(0, z_max+0.1, 0.1)

    vol = volume_effective_Gpcperh(z_arr, z_bins, cosmo, fsky, P0)
    vol_fmt = rf'{vol:.2f} $(h^{{-1}}\,Gpc)^3$'
    quants[f'volume_effective_{name}'] = vol_fmt
    
    z_range = [0.8, 2.2]
    vol_span = v_shells_of_z(z_range, fsky, cosmo)[0]
    vol_span = (vol_span.to(u.Gpc**3) * cosmo.h**3).value # (Gpc) -> (Gpc/h)^3, mult by h^3 
    vol_span_fmt = rf'{vol_span:.2f} $(h^{{-1}}\,Gpc)^3$'
    print('Spanning volume is', vol_span_fmt)
    quants[f'volume_spanning_{name}'] = vol_span_fmt
        
    N_arr.append(f'{N:,}')
    area_arr.append(f'{area:.2f}')
    nbar_arr.append(f'{nbar:.2f}')
    fsky_arr.append(f'{fsky:.2f}')
    veff_arr.append(f'{vol:.2f}')
    vspan_arr.append(f'{vol_span:.2f}')
    
    #zmed_arr.append(f'{zmed:.2f}')
     #     print(quants[f'N_{name}'])
#     print(quants[f'area_{name}'])
#     print(quants[f'nbar_{name}'])
#     print(quants[f'fsky_{name}'])
#     print(quants[f'volume_effective_{name}'])
#     print(quants[f'volume_spanning_{name}'])
#     print(quants[f'zmed_{name}'])
#     print()

Area from 15436/49152 pixels
nbar = 108.13
Sky fraction is 0.314
Effective volume is 38.539 Gpc3 = 11.937 (Gpc/h)^3
Spanning volume is 61.53 $(h^{-1}\,Gpc)^3$


In [19]:
len(tab_desi_good[(tab_desi_good['Z']>0.8) & (tab_desi_good['Z']<3.1)])

1264943